# Weekly successful trials: VRDN and mechanosensation

Each screen has a plot with **all mapped lines on the x axis**, including lines with zero successful trials. Colours show the contribution from each week; total bar height is cumulative success. Names come from `condition_2`, matching conditions without case or whitespace differences. Intermediate sheets are not required.

Only `amendments = 0` is successful and `-1` is failed. Blank or unrecognised amendments are unscored. Set `BLANK_IS_SUCCESSFUL = True` only if blank amendments in your completed weekly sheets mean successful trials. No pending counts are inferred from missing intermediate sheets.

Run this notebook from the repository or `notebooks` directory. Set each `weekly_dir` to its current source folder. The lab-drive paths below match `exp-stats.ipynb`; local mechanosensation sheets are used when that drive is unavailable. Those local sheets cover only weeks 1–2, so their plot is a partial view. CSV and every experiment tab in Excel workbooks are read. Keep only one copy of each week's trials in each folder to avoid double counting.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path.cwd()
if not (REPO / "scripts" / "weekly_screen_stats.py").exists():
    REPO = REPO.parent
if not (REPO / "scripts" / "weekly_screen_stats.py").exists():
    raise FileNotFoundError("Run from the screen-stats repository or its notebooks folder")
sys.path.insert(0, str(REPO))
from scripts.weekly_screen_stats import load_mapping, read_weekly_sheets, summarise, plot_successful

LAB_REPO = Path("/Volumes/lab-windingm/home/shared/repos/screen-stats")
BLANK_IS_SUCCESSFUL = False
TARGET = 18
OUTPUT = REPO / "results" / "weekly-screen-stats"
OUTPUT.mkdir(parents=True, exist_ok=True)

SCREENS = {
    "VRDN-screen": {
        "weekly_dir": LAB_REPO / "VRDN-screen",
        "mapping_csv": (LAB_REPO / "fly-lines" / "VRDN-stocks_mapping.csv"
                        if (LAB_REPO / "fly-lines" / "VRDN-stocks_mapping.csv").exists()
                        else REPO / "notebooks" / "condition_mapping.csv"),
    },
    "mechano-screen": {
        "weekly_dir": (LAB_REPO / "mechano-screen" if (LAB_REPO / "mechano-screen").exists()
                       else REPO / "mechano-screen"),
        "mapping_csv": REPO / "notebooks" / "condition_mapping.csv",
    },
}


In [ ]:
results = {}
for screen, settings in SCREENS.items():
    print(f"\n{screen}: {settings['weekly_dir']}")
    if not settings["weekly_dir"].is_dir():
        print("SOURCE UNAVAILABLE: set weekly_dir to your weekly sheets folder, then rerun.")
        continue
    mapping = load_mapping(settings["mapping_csv"], screen)
    actual = read_weekly_sheets(settings["weekly_dir"])
    counts, totals, unmapped = summarise(actual, mapping, blank_is_successful=BLANK_IS_SUCCESSFUL)
    results[screen] = {"counts": counts, "totals": totals, "actual": actual}
    print(f"{len(actual)} trial rows; {len(totals)} mapped/observed lines; weeks: "
          + ", ".join(sorted(actual["week"].unique(), key=float)))
    print(totals[["successful", "failed", "unscored"]].sum().to_string())
    if unmapped:
        print("Unmapped conditions (shown using raw names):", ", ".join(unmapped))
    display(totals)
    display(counts.groupby("week", sort=False)[["successful", "failed", "unscored"]].sum())
    title = f"{screen}: successful trials by mapped line and week"
    if settings["weekly_dir"] == REPO / "mechano-screen":
        title += " (local weeks 1–2 only)"
    if totals["unscored"].sum():
        title += f"\n{totals['unscored'].sum()} unscored trials excluded from success counts"
    fig, ax = plot_successful(counts, title, target=TARGET)
    fig.savefig(OUTPUT / f"{screen}-successful.png", dpi=160, bbox_inches="tight")
    fig.savefig(OUTPUT / f"{screen}-successful.pdf", bbox_inches="tight")
    counts.to_csv(OUTPUT / f"{screen}-weekly-counts.csv", index=False)
    totals.to_csv(OUTPUT / f"{screen}-line-totals.csv", index=False)
    plt.show()
    plt.close(fig)


The exported PNG/PDF figures and CSV tables are in `results/weekly-screen-stats`. A missing source folder is reported explicitly; it is never treated as a screen with zero successes. Blank amendments in source rows are kept separate from mapped lines with no trial rows.
